# 第16篇｜数据透视表与交叉分析：一行代码做多维报表

> 这是「数据分析从入门到精通」系列的第 16 篇。多表合并搞定了，接下来这篇聊数据分析中最常用的"汇报利器"——数据透视表。一行代码，多维切片，老板看了会说"就要这种感觉"。

---

嗨，我是小荷～

上一篇我们把三张表合并成了一张大宽表，现在数据齐了，但老板说："我要看每个城市、每个品类的月度销售对比。"

你打开 Excel，想用鼠标拖透视表……等等，我们是学 Pandas 的！

数据透视表是多维分析最强的武器之一。当年萧何整理全国税赋数据，要是有 `pivot_table`，估计能省出一半的时间来——按郡、按年、按品类，一行代码就出来了。

---

## 一、pivot_table 基础语法

先来看最常用的语法格式：


In [ ]:
pd.pivot_table(
    data,           # 数据源 DataFrame
    values,         # 要计算的数值列
    index,          # 行维度（相当于行标签）
    columns,        # 列维度（相当于列标签）
    aggfunc,        # 聚合函数，默认 mean
    fill_value,     # 填充 NaN 的值
    margins,        # 是否显示汇总行列（All）
    margins_name    # 汇总行列的名称
)


### 最简单的例子

来看看最简单的例子怎么实现：


In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

df = pd.DataFrame({
    'city':     np.random.choice(['北京', '上海', '广州', '深圳'], n),
    'category': np.random.choice(['数码', '服装', '食品', '美妆'], n),
    'month':    np.random.choice(['1月', '2月', '3月'], n),
    'sales':    np.random.randint(100, 1000, n),
    'quantity': np.random.randint(1, 20, n)
})

# 各城市、各品类的平均销售额
pivot1 = pd.pivot_table(df, values='sales', index='city', columns='category', aggfunc='mean').round(1)
print(pivot1)


category     数码     服装     美妆     食品
city                                
上海        590.1  338.4  650.2  588.2
北京        616.2  584.1  524.4  547.1
广州        544.6  516.8  546.3  470.2
深圳        645.9  579.8  519.4  586.1


输出（示意）：


In [ ]:
category    数码    服装    食品    美妆
city
上海       521.3  489.2  503.7  478.6
北京       498.7  512.0  487.3  501.2
广州       507.4  495.8  512.6  498.0
深圳       489.2  508.3  499.4  521.7


这就是交叉分析！行是城市，列是品类，格子里是平均销售额。

---

## 二、aggfunc 聚合函数详解

`aggfunc` 不只能求均值，常用的有：


In [2]:
# 求和
pivot_sum = pd.pivot_table(df, values='sales', index='city', columns='category', aggfunc='sum')

# 计数
pivot_count = pd.pivot_table(df, values='sales', index='city', columns='category', aggfunc='count')

# 最大值
pivot_max = pd.pivot_table(df, values='sales', index='city', columns='category', aggfunc='max')

# 同时计算多个指标
pivot_multi = pd.pivot_table(
    df,
    values='sales',
    index='city',
    columns='category',
    aggfunc=['sum', 'mean', 'count']
)
print(pivot_multi)


           sum                           mean                          \
category    数码    服装     美妆    食品          数码          服装          美妆   
city                                                                    
上海        6491  3046  10404  5882  590.090909  338.444444  650.250000   
北京        7395  4089   9440  4924  616.250000  584.142857  524.444444   
广州        7080  6201   6556  7993  544.615385  516.750000  546.333333   
深圳        8397  6957   7791  8205  645.923077  579.750000  519.400000   

                     count              
category          食品    数码  服装  美妆  食品  
city                                    
上海        588.200000    11   9  16  10  
北京        547.111111    12   7  18   9  
广州        470.176471    13  12  12  17  
深圳        586.071429    13  12  15  14  


---

## 三、多维分析：index 和 columns 可以是列表

多维度交叉分析，才是透视表的正确打开方式：


In [ ]:
# 行：城市 + 月份，列：品类
pivot_3d = pd.pivot_table(
    df,
    values='sales',
    index=['city', 'month'],   # 行有两个层次
    columns='category',
    aggfunc='sum',
    fill_value=0               # NaN 填 0
)
print(pivot_3d)


---

## 四、margins 参数：自动加合计行列

加个合计行列，报表更完整：


In [3]:
pivot_margin = pd.pivot_table(
    df,
    values='sales',
    index='city',
    columns='category',
    aggfunc='sum',
    fill_value=0,
    margins=True,              # 开启合计
    margins_name='合计'        # 合计行/列的名字
)
print(pivot_margin)


category     数码     服装     美妆     食品      合计
city                                        
上海         6491   3046  10404   5882   25823
北京         7395   4089   9440   4924   25848
广州         7080   6201   6556   7993   27830
深圳         8397   6957   7791   8205   31350
合计        29363  20293  34191  27004  110851


输出：


In [ ]:
category    数码    服装    食品    美妆    合计
city
上海       12300   9800  10500  8900   41500
北京       11200  10300   9700  9200   40400
广州       10800   9500  11200  8700   40200
深圳       11500  10100   9900  9500   41000
合计       45800  39700  41300 36300  163100


这种格式可以直接贴给老板看 👆

---

## 五、crosstab：快速交叉频率统计

`pd.crosstab()` 是 `pivot_table` 的特化版，专门用于**频率/计数统计**：


In [ ]:
# 各城市 × 各品类 的订单次数
cross = pd.crosstab(df['city'], df['category'])
print(cross)

# 加上百分比（normalize='index' 按行归一化）
cross_pct = pd.crosstab(df['city'], df['category'], normalize='index').round(3) * 100
print(cross_pct)


输出（百分比版示意）：


In [ ]:
category  数码   服装   食品   美妆
city
上海      26.0  22.0  27.0  25.0
北京      25.0  24.0  23.0  28.0
...


这就是"各城市用户中，哪个品类最受欢迎"的答案。

---

## 六、pivot vs melt：宽表与长表互转

透视表让数据变"宽"（每个品类是一列），有时候我们反而需要"长"格式。


In [ ]:
# 宽表 → 长表（melt）
wide_df = pd.DataFrame({
    'city': ['北京', '上海', '广州'],
    '数码':  [12300, 11200, 10800],
    '服装':  [9800,  10300, 9500]
})

long_df = wide_df.melt(id_vars='city', var_name='category', value_name='sales')
print(long_df)


输出：


In [ ]:
   city category  sales
0    北京       数码  12300
1    上海       数码  11200
2    广州       数码  10800
3    北京       服装   9800
4    上海       服装  10300
5    广州       服装   9500


**长表适合画图，宽表适合展示。** 根据场景选合适的格式。

---

## 七、🔧 综合实战：制作多维经营分析报表

学了一堆理论，来个完整的实战练练手——把前面学的知识点串起来：


In [5]:
import pandas as pd
import numpy as np

# ── 构造电商数据 ──
np.random.seed(123)
n = 500

df = pd.DataFrame({
    'order_date': pd.date_range('2024-01-01', periods=n, freq='15h'),
    'city':       np.random.choice(['北京', '上海', '广州', '深圳', '杭州'], n, p=[0.25, 0.25, 0.2, 0.15, 0.15]),
    'category':   np.random.choice(['数码', '服装', '食品', '美妆', '图书'], n),
    'user_level': np.random.choice(['普通', '银卡', '金卡', '钻石'], n, p=[0.4, 0.3, 0.2, 0.1]),
    'sales':      np.random.randint(50, 2000, n),
    'profit':     np.random.randint(10, 500, n)
})
df['month'] = df['order_date'].dt.month.map({1: '1月', 2: '2月', 3: '3月'})

# ── 报表1：各城市各品类销售额（附合计）──
print("=" * 60)
print("📊 报表1：各城市 × 各品类 销售额汇总")
print("=" * 60)
report1 = pd.pivot_table(
    df, values='sales', index='city', columns='category',
    aggfunc='sum', fill_value=0, margins=True, margins_name='合计'
)
print(report1)

# ── 报表2：各月各品类平均客单价 ──
print("\n" + "=" * 60)
print("📊 报表2：各月 × 各品类 平均客单价")
print("=" * 60)
report2 = pd.pivot_table(
    df, values='sales', index='month', columns='category',
    aggfunc='mean', fill_value=0
).round(1)
print(report2)

# ── 报表3：会员等级 × 城市 的利润贡献 ──
print("\n" + "=" * 60)
print("📊 报表3：会员等级 × 城市 利润贡献")
print("=" * 60)
report3 = pd.pivot_table(
    df, values='profit', index='user_level', columns='city',
    aggfunc='sum', fill_value=0, margins=True, margins_name='合计'
)
# 按合计列排序
report3 = report3.sort_values('合计', ascending=False)
print(report3)

# ── 报表4：各城市品类偏好（百分比）──
print("\n" + "=" * 60)
print("📊 报表4：各城市品类偏好分布 (%)")
print("=" * 60)
report4 = pd.crosstab(df['city'], df['category'], normalize='index').round(3) * 100
print(report4)

# ── 找出最畅销的城市+品类组合 ──
top_combo = pd.pivot_table(df, values='sales', index=['city', 'category'],
                            aggfunc='sum').sort_values('sales', ascending=False)
print("\n🏆 销售额 TOP 5 城市+品类组合：")
print(top_combo.head())


📊 报表1：各城市 × 各品类 销售额汇总
category      图书      数码     服装      美妆     食品      合计
city                                                  
上海         22360   41648  25372   26257  23327  138964
北京         28062   34197  14119   37394  18527  132299
广州         20003    8500  18608   30474  20374   97959
杭州         15305   19678  19440    9833  14373   78629
深圳         16607    9847  19724   12077  17285   75540
合计        102337  113870  97263  116035  93886  523391

📊 报表2：各月 × 各品类 平均客单价
category      图书      数码      服装      美妆      食品
month                                           
1月        1467.6  1006.2   981.2   910.4   969.0
2月        1171.0  1017.4  1012.4  1200.2   738.4
3月        1210.8  1244.4   860.6  1115.1  1153.5

📊 报表3：会员等级 × 城市 利润贡献
city           上海     北京     广州     杭州     深圳      合计
user_level                                           
合计          32962  29149  25481  16629  17601  121822
普通          12314  15104  13364   7328   8712   56822
银卡          10043   6598   6541  

跑一下这段代码，你就能得到一份完整的多维经营分析报表——这种报表在真实工作中每天都要用到。

---

## 八、📋 速查表

| 需求 | 方法 |
|------|------|
| 多维均值/求和 | `pd.pivot_table(aggfunc='mean'/'sum')` |
| 加合计行列 | `margins=True, margins_name='合计'` |
| 频率/计数交叉表 | `pd.crosstab()` |
| 按行百分比 | `pd.crosstab(normalize='index')` |
| 宽表转长表 | `df.melt(id_vars=..., var_name=..., value_name=...)` |
| NaN 填 0 | `fill_value=0` |

---

## 九、📝 小结

数据透视表的本质是：**把数据按两个维度重新排列，用聚合函数计算格子里的值**。

- `pivot_table`：灵活、功能全，任何场景都适用
- `crosstab`：专注于频率/计数，语法更简洁
- `melt`：透视的逆操作，宽转长，为画图做准备

---

## 十、🏋️ 课后练习

1. 用你之前整合的三表大宽表，做一个"各会员等级 × 各品类"的订单数透视表。
2. 在透视表里加上 `margins=True`，看看哪个品类的总订单量最高。
3. 用 `pd.crosstab` 计算各城市中金卡/钻石会员的占比，找出高价值用户最集中的城市。

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 17 篇：apply / map / transform 自定义处理**
>
> 数据筛选、分组、合并、透视都学会了——但有些操作，用这些方法写起来太繁琐。下篇介绍 apply / map / transform，让你用自定义函数处理任何复杂的数据变换，一行代码解决「批量打标签」这类需求。

---

*跟着小荷，数据分析路上不迷路～*
